# ml_tool 使用示例

完整流程：特征分析 → 分箱 → 特征筛选 → 模型训练 → 报告输出

In [ ]:
import pandas as pd
import numpy as np
from ml_tool import FeatureAnalyzer, Binning, FeatureSelector, ModelTrainer, ReportGenerator

# 构造示例数据（替换为真实数据）
np.random.seed(42)
n = 3000
df = pd.DataFrame({
    "f1": np.random.randn(n),
    "f2": np.random.rand(n),
    "f3": np.random.randn(n) * 2,
    "f4": np.random.rand(n),
    "f5": np.random.randn(n),
    "y":  np.random.binomial(1, 0.15, n),
    "dataset": np.random.choice(["train","test","oot"], n, p=[0.6, 0.2, 0.2]),
    # month 列：实际使用时替换为真实月份字段（如 df["ym"]）
    "month": np.random.choice(["2024-01","2024-02","2024-03"], n),
})
# 注入部分缺失值
df.loc[df.sample(frac=0.05).index, "f2"] = np.nan
print(df.shape, df["dataset"].value_counts().to_dict())
print("月份分布:", df["month"].value_counts().sort_index().to_dict())

## 1. 特征分析

In [ ]:
feature_cols = ['f1', 'f2', 'f3', 'f4', 'f5']
analyzer = FeatureAnalyzer(df[feature_cols])

print('=== 缺失率 ===')
display(analyzer.missing_rate())

print('=== 一值率 ===')
display(analyzer.single_value_rate())

print('=== 分位数统计 ===')
display(analyzer.quantile_stats())

In [ ]:
# PSI：以 train 为基准组，oot 为对照组
base_df    = df[df['dataset'] == 'train'][feature_cols]
compare_df = df[df['dataset'] == 'oot'][feature_cols]
psi_df = analyzer.psi(base_df, compare_df)
print('=== PSI（train vs OOT）===')
display(psi_df)

# 相关性（返回高相关特征对）
print('=== 特征相关性（|r|≥0.3）===')
display(analyzer.correlation(threshold=0.3))

## 2. 分箱与 WOE/IV 计算

In [ ]:
train_df = df[df['dataset'] == 'train'].reset_index(drop=True)

binning = Binning(n_bins=10)
binning.fit(train_df[feature_cols + ['y']], target='y', cols=feature_cols)

print('=== IV 汇总 ===')
display(binning.get_iv_summary())

print('=== f1 分箱详情 ===')
display(binning.get_iv_details('f1'))

In [ ]:
# 将原始特征替换为等频分箱编号（mode='bin_index'）或 WOE 值（mode='woe'）
df_binned = binning.transform(df[feature_cols + ['y', 'dataset']], cols=feature_cols, mode='bin_index')
print('分箱转换后（前5行）:')
display(df_binned.head())

## 3. 特征筛选

In [ ]:
selector = FeatureSelector(
    iv_threshold=0.001,
    psi_threshold=0.5,
    missing_threshold=0.98,
    corr_threshold=0.97,
)
selector.fit(
    df=train_df[feature_cols + ['y']],
    target='y',
    base_df=base_df,
    compare_df=compare_df,
)

print('保留特征:', selector.get_selected())
print('剔除特征:', selector.dropped_cols)
display(selector.get_report())

In [ ]:
# 输出特征筛选 Excel 报告（含缺失率、PSI、IV、相关性等全量数据）
reporter = ReportGenerator(output_dir='./reports')
analysis = {
    '缺失率': analyzer.missing_rate(),
    '分位数统计': analyzer.quantile_stats(),
    'PSI': psi_df,
    '相关性': analyzer.correlation(threshold=0.0),
    'IV汇总': binning.get_iv_summary(),
}
path = reporter.feature_selection_report(selector.get_report(), analysis_results=analysis)
print('特征筛选报告已保存：', path)

## 4. 模型训练（LightGBM）

In [ ]:
selected = selector.get_selected()

X_train = df[df['dataset']=='train'][selected]
y_train = df[df['dataset']=='train']['y']
X_test  = df[df['dataset']=='test'][selected]
y_test  = df[df['dataset']=='test']['y']
X_oot   = df[df['dataset']=='oot'][selected]
y_oot   = df[df['dataset']=='oot']['y']

# n_trials 设小以便快速演示，实际建议 lgbm=150, xgboost=100
trainer = ModelTrainer(model_type='lgbm', n_trials=20)
trainer.fit(X_train, y_train, X_test, y_test, X_oot, y_oot)

print('最终入模特征数:', len(trainer.selected_features))
print('最优参数:', trainer._trainer.best_params)

## 5. 分类模型评估报告（含 Lift / 样本占比 / 按月 KS / 按月分箱）

In [ ]:
# 构造 month_col_data：每个数据集对应的月份列 DataFrame，行与 y_true 对齐
month_data_clf = {
    "训练集": df[df["dataset"]=="train"][["month"]].reset_index(drop=True),
    "测试集": df[df["dataset"]=="test"][["month"]].reset_index(drop=True),
    "OOT":   df[df["dataset"]=="oot"][["month"]].reset_index(drop=True),
}

datasets_clf = {
    "训练集": (y_train.values, trainer.predict_proba(X_train)),
    "测试集": (y_test.values,  trainer.predict_proba(X_test)),
    "OOT":   (y_oot.values,   trainer.predict_proba(X_oot)),
}

clf_result = reporter.classification_report(
    datasets_clf,
    filename="LightGBM_分类评估报告",
    month_col_data=month_data_clf,
)

display(clf_result["summary"])
print("生成 sheets:", list(clf_result["sheets"].keys()))
print("Excel:", clf_result["excel_path"])
print("HTML: ", clf_result["html_path"])

# 查看分箱表：含 Lift 和样本占比
display(clf_result["sheets"]["训练集_分箱分析"])

## 6. 回归模型评估报告（含 MAPE / 样本占比 / 按月分桶）

In [ ]:
# 回归模型训练
y_train_reg = df[df["dataset"]=="train"]["f3"]
y_test_reg  = df[df["dataset"]=="test"]["f3"]
y_oot_reg   = df[df["dataset"]=="oot"]["f3"]

trainer_reg = ModelTrainer(model_type="xgboost_reg", n_trials=20)
trainer_reg.fit(X_train, y_train_reg, X_test, y_test_reg, X_oot, y_oot_reg)

# 构造月份数据
month_data_reg = {
    "训练集": df[df["dataset"]=="train"][["month"]].reset_index(drop=True),
    "测试集": df[df["dataset"]=="test"][["month"]].reset_index(drop=True),
    "OOT":   df[df["dataset"]=="oot"][["month"]].reset_index(drop=True),
}

datasets_reg = {
    "训练集": (y_train_reg.values, trainer_reg.predict(X_train)),
    "测试集": (y_test_reg.values,  trainer_reg.predict(X_test)),
    "OOT":   (y_oot_reg.values,   trainer_reg.predict(X_oot)),
}

reg_result = reporter.regression_report(
    datasets_reg,
    filename="XGBoost_回归评估报告",
    month_col_data=month_data_reg,
)

display(reg_result["summary"])
print("生成 sheets:", list(reg_result["sheets"].keys()))
print("Excel:", reg_result["excel_path"])
print("HTML: ", reg_result["html_path"])

# 查看分桶表：含 MAPE 和样本占比
display(reg_result["sheets"]["训练集_分桶分析"])

## 6-A. LightGBM 回归训练（lgbm_reg）

与 xgboost_reg 使用相同的调参 loss，两轮 Hyperopt + gain 特征筛选。

In [ ]:
# LightGBM 回归训练，loss 函数与 xgboost_reg 完全一致：
#   oot_mae + 0.5*(oot_mae/tr_mae) + 0.3*|ts_mae-oot_mae|/ts_mae
trainer_lgbm_reg = ModelTrainer(model_type="lgbm_reg", n_trials=20)
trainer_lgbm_reg.fit(
    X_train, y_train_reg,
    X_test,  y_test_reg,
    X_oot,   y_oot_reg,
)

print("入模特征数:", len(trainer_lgbm_reg.selected_features))
print("入模特征:", trainer_lgbm_reg.selected_features)
print("objective:", trainer_lgbm_reg.best_params.get("objective"))
print("metric:",    trainer_lgbm_reg.best_params.get("metric"))

# 对比三组 MAE（trials_log 里的 oot_mae 是调参 loss 中的绝对值项）
best_row = trainer_lgbm_reg.trials_log.loc[trainer_lgbm_reg.trials_log["loss"].idxmin()]
print("best trial MAE  train/test/oot:")
print("  train_mae=", best_row["train_mae"],
      " test_mae=",  best_row["test_mae"],
      " oot_mae=",   best_row["oot_mae"],
      " loss=",      best_row["loss"])

## 7. 第二轮需求 — 不重启修改参数空间（custom_space）

在 Jupyter 中无需重启 kernel，直接修改参数范围后传入  重新训练。

In [ ]:
from hyperopt import hp
from ml_tool import ModelTrainer

# 第一步：获取默认参数空间
trainer2 = ModelTrainer(model_type="lgbm", n_trials=20)
space = trainer2.get_default_space(n_data=len(X_train))

# 第二步：在 notebook 里直接修改想调整的参数范围，无需重启
space["learning_rate"] = hp.uniform("learning_rate", 0.001, 0.05)   # 缩小学习率范围
space["max_depth"]     = hp.quniform("max_depth", 2, 3, 1)          # 限制最大深度
space["n_estimators"]  = hp.quniform("n_estimators", 200, 400, 10)  # 减少迭代轮次加快速度

# 第三步：传入 custom_space 开始训练
trainer2.fit(X_train, y_train, X_test, y_test, X_oot, y_oot, custom_space=space)

print("入模特征数:", len(trainer2.selected_features))
print("实际 learning_rate:", trainer2.best_params.get("learning_rate"))
print("实际 max_depth:", trainer2.best_params.get("max_depth"))

## 8. 第二轮需求 — 保存模型 & 加载恢复

In [ ]:
import os

# 保存模型（训练时也可直接传 save_dir 参数一步完成）
trainer2.save("./saved_models/lgbm_demo")

print("保存的文件:")
for f in sorted(os.listdir("./saved_models/lgbm_demo")):
    print(" ", f)

# 加载恢复，自动识别模型类型
loaded_trainer = ModelTrainer.load("./saved_models/lgbm_demo")
print("加载成功，入模特征:", loaded_trainer.selected_features)

# 验证预测一致性
import numpy as np
pred_orig   = trainer2.predict_proba(X_oot)
pred_loaded = loaded_trainer.predict_proba(X_oot)
print("预测最大误差:", round(float(np.abs(pred_orig - pred_loaded).max()), 8))

## 9. 第二轮需求 — 按分组列输出 KS / AUC

支持按任意列（如 、月份列）分组展示模型表现。

In [ ]:
from ml_tool import evaluate_clf_by_group
import numpy as np

# 把预测分数拼到全量数据
df_eval = df.copy()
df_eval["score"] = loaded_trainer.predict_proba(df[selected])

# 按 dataset 分组输出 KS/AUC
print("=== 按 dataset 分组 KS/AUC ===")
result_ds = evaluate_clf_by_group(
    df_eval, group_col="dataset", y_col="y", score_col="score"
)
display(result_ds)

# 模拟月份列，按月分组
np.random.seed(0)
df_eval["month"] = np.random.choice(["2024-01","2024-02","2024-03"], len(df_eval))
print("=== 按 month 分组 KS/AUC ===")
result_month = evaluate_clf_by_group(
    df_eval, group_col="month", y_col="y", score_col="score"
)
display(result_month)

---
## 第三轮新增功能


## 10. 第三轮 — 分类报告按月 KS / 按月分箱明细

In [ ]:
# 按月 KS/AUC 汇总
print("=== 按月 KS/AUC ===")
display(clf_result["sheets"]["按月KS_AUC"])

# 训练集按月分箱明细（每月 x 每箱 的 Lift / 样本占比 / 坏样本率）
print("=== 训练集按月分箱明细 ===")
display(clf_result["sheets"]["训练集_按月分箱"])

## 11. 第三轮 — 回归报告 MAPE / 按月分桶明细

In [ ]:
# 汇总表中的 MAPE 列
print("=== 回归汇总（含 MAPE）===")
display(reg_result["summary"][["数据集","RMSE","MAE","R2","MAPE(%)","样本量"]])

# 分桶表：含 MAPE + 样本占比
print("=== 训练集分桶（含 MAPE + 样本占比）===")
display(reg_result["sheets"]["训练集_分桶分析"])

# 按月分桶明细
print("=== 训练集按月分桶明细 ===")
display(reg_result["sheets"]["训练集_按月分桶"])

## 12. 第三轮 — 特征分箱报告（按 dataset 输出各箱坏样本率 + 样本占比）

In [ ]:
# feature_bin_report：按 dataset 分别输出每个特征的分箱统计
# 每个特征一个 sheet，包含：样本数、样本占比、坏样本数、坏样本率、坏样本在全集占比
bin_report_path = reporter.feature_bin_report(
    df=df,
    target="y",
    features=feature_cols,
    dataset_col="dataset",
    binning=binning,
    filename="特征分箱报告_按dataset",
)
print("特征分箱报告路径:", bin_report_path)

import openpyxl
wb = openpyxl.load_workbook(bin_report_path)
print("生成的 sheet:", wb.sheetnames)

# 预览 f1 的分箱结果（train / test / oot 各自的坏样本率和样本占比）
import pandas as pd
f1_df = pd.read_excel(bin_report_path, sheet_name="f1")
print("=== f1 分箱统计（train / test / oot）===")
display(f1_df)

## 13. 第三轮 — 特征分箱报告（不传 binning，自动拟合）

In [ ]:
# 不传 binning 参数时，自动用 dataset=="train" 的数据拟合分箱边界
bin_report_auto = reporter.feature_bin_report(
    df=df,
    target="y",
    features=["f1", "f3", "f5"],
    dataset_col="dataset",
    binning=None,
    n_bins=10,
    filename="特征分箱报告_自动拟合",
)
print("自动拟合分箱报告路径:", bin_report_auto)

# 对比 train vs test vs oot 的坏样本率变化
import pandas as pd
df_f3 = pd.read_excel(bin_report_auto, sheet_name="f3")
pivot = df_f3.pivot_table(index="分箱区间", columns="dataset", values="坏样本率", aggfunc="first")
print("=== f3 各箱坏样本率：train vs test vs oot ===")
display(pivot)